# GP-WLS: Gradient Ascent with Wolfe Line Search

This notebook explains how GP-WLS optimizes the GP posterior mean, and why Wolfe conditions replace the fixed termination parameter used in related work.

## 1.  The optimization problem

We want to maximise an unknown, expensive-to-evaluate function $f : \mathcal{X} \to \mathbb{R}$:

$$x^* = \arg\max_{x \in \mathcal{X}} f(x)$$

Each evaluation of $f(x)$ is expensive, so we use a **GP surrogate** $\mu(x)$ fitted to the observed data. Instead of evaluating $f$ everywhere, we optimize $\mu(x)$ , which is cheap, and use that to decide where to evaluate $f$ next.

### Algorithm overview

```
1. Sample n_init random points, evaluate f, fit GP
2. Optimize GP hyperparameters via LML
3. Repeat for n_iter iterations:
     a. Inner loop: gradient ascent on mu(x) -->  candidate x_cand
     b. Evaluate f(x_cand)  (one real evaluation)
     c. Update dataset (sliding window, N_MAX = 30)
     d. Refit GP and optimize hyperparameters
4. Return best observed point
```

The key idea: **one real function evaluation per outer iteration**, guided by the cheap inner optimization of $\mu(x)$.

## 2.  Posterior gradient

The inner loop performs gradient ascent on the posterior mean $\mu(x)$. We need $\nabla_x \mu(x)$.

Recall that:

$$\mu(x) = K(x, X)\, \alpha, \qquad \alpha = (K + \sigma_n^2 I)^{-1} \mathbf{y}$$

The gradient with respect to $x$ flows through the kernel function:

$$\nabla_x \mu(x) = \nabla_x K(x, X)\, \alpha$$

For the Matern $\nu=5/2$ kernel this derivative exists and is continuous everywhere (the kernel is twice differentiable). In our implementation, we compute this via **PyTorch autograd**. Therefore, no manual derivation is needed. Autograd traces the computation graph through $K(x, X) \to \alpha \to \mu(x)$ and returns the exact gradient.

The gradient is then used as the ascent direction in the update:

$$x_{t+1} = x_t + \alpha_t \cdot \nabla_x \mu(x_t)$$

## 3.  Inspiration: GIBO and the termination problem

GP-WLS is inspired by **GIBO** (Gradient Information with Bayesian Optimization) by Müller et al. (2021) [1], which also uses the GP posterior mean inside a Bayesian optimization loop and a Gradient Information aquisition function.

GIBO terminates the inner loop after a **fixed number of steps $M$**:

```
for m in range(M):          #M is a fixed hyperparameter
    g = posterior_gradient(gp, x)
    x = x + alpha * g
```

$M$ is a manual tuning parameter with no principled default:

| | $M$ too small | $M$ too large |
|---|---|---|
| Effect | Inner loop stops before converging | Inner loop continues past convergence |
| Consequence | Suboptimal candidate $x_{\text{cand}}$ | Wasted gradient evaluations |

GP-WLS investigates whether replacing $M$ with a **principled adaptive criterion** removes this manual tuning while maintaining convergence quality.

> [1] Müller, S., von Rohr, A., & Trimpe, S. (2021). *Local policy search with Bayesian optimization.* Advances in Neural Information Processing Systems (NeurIPS), 34.

## 4.  Wolfe conditions

At each inner loop step we search for a step size $\alpha$ along the gradient direction $d = \nabla_x \mu(x)$. Define the 1D slice of the posterior mean along this direction:

$$\phi(\alpha) = \mu(x + \alpha d), \qquad \phi'(\alpha) = \nabla_x \mu(x + \alpha d)^\top d$$

A step size $\alpha > 0$ satisfies the **strong Wolfe conditions** if:

**Armijo condition** (sufficient increase):
$$\phi(\alpha) \geq \phi(0) + c_1\, \alpha\, \phi'(0)$$

**Curvature condition**:
$$|\phi'(\alpha)| \leq c_2\, |\phi'(0)|$$

with $0 < c_1 < c_2 < 1$, typically $c_1 = 10^{-4}$ and $c_2 = 0.9$.

**Armijo** says: the step must increase $\phi$ by at least fraction $c_1$ of what a linear model predicts. This rules out steps that are negligibly small.

**Curvature** says: the directional derivative at the new point must be smaller in magnitude than at the start —-> we moved toward a flatter region, i.e., toward a maximum of $\phi$.

When both conditions hold, we are at a point where:
- $\mu$ increased sufficiently (Armijo), and
- the gradient is small enough that further steps yield diminishing returns (curvature).

The inner loop then terminates when $\|\nabla_x \mu(x)\| < \varepsilon$ —-> the gradient is small, indicating we are near a local maximum of $\mu$. The number of steps is determined entirely by the geometry of $\mu$, not by a fixed parameter.

## 5.  Bracket and zoom

Finding a valid $\alpha$ follows the standard procedure from Nocedal & Wright, *Numerical Optimization* (2006), Chapter 3:

**Phase 1 — Bracket:** Starting from a small $\alpha_0$, increase $\alpha$ (e.g., doubling) until either:
- The Armijo condition is violated (we stepped too far), or
- The function value stopped increasing (we passed a maximum).

This yields a bracket $[\alpha_{\text{lo}}, \alpha_{\text{hi}}]$ guaranteed to contain a Wolfe-satisfying $\alpha$.

**Phase 2 — Zoom:** Bisect the bracket, check conditions at the midpoint, shrink the bracket accordingly. Converges in a small number of evaluations.

### Why not SciPy?

`scipy.optimize.line_search_wolfe` operates on **NumPy arrays**. Every call to $\phi(\alpha)$ would require a NumPy–PyTorch conversion, which **breaks the autograd graph** --> the gradient $\nabla_x \mu(x)$ with respect to $x$ would no longer be traceable through the computation. The Wolfe line search must run entirely in PyTorch to preserve autograd compatibility.

## 6.  Input normalization

All inputs are normalized to $[0, 1]^d$ before fitting the GP:

$$x_{\text{norm}} = \frac{x - x_{\text{low}}}{x_{\text{high}} - x_{\text{low}}}$$

This ensures the length scale $\ell$ of the Matern kernel is interpretable (a length scale of 1 means the function varies on the scale of the full input range) and avoids near-zero kernel values that would make the Cholesky factorization bad conditioned.

## 7.  Sliding window

Cholesky factorization costs $O(n^3)$ in the number of training points. To bound this cost, GP-WLS keeps only the **$N_{\text{MAX}} = 30$ most recent observations** in the GP training set --> older points are discarded.

This is appropriate for local search: recent observations cluster near the current optimum and are more informative than early exploratory points. The factorization cost stays constant at $O(30^3)$ regardless of total iterations.